# Chain

**Chain**(체인)은 여러 컴포넌트(요소)를 정해진 순서대로 연결하여 **복잡한 AI 작업을 단계별로 자동화**할 수 있도록 돕는 구조이다.

- 각 컴포넌트는 **이전 처리결과를 입력으로 받아 처리한 후 다음 단계로 결과를 전달**한다.
- 복잡한 작업을 여러 개의 단순한 단계로 나누고, 각 단계를 순차적으로 실행함으로써 전체 작업을 체계적으로 구성할 수 있다.

## 기본 개념

- 체인은 하나의 LLM 호출에 그치지 않고 **여러 LLM 호출이나 도구 실행등을 순차적으로 연결**하여 실행 할 수 있다.
- 예를 들어, 사용자의 질문 → 검색 → 요약 → 응답 생성 같은 일련의 작업을 체인으로 구성할 수 있다.
- 이러한 체인구조를 사용하면 **작업흐름이 명확**해지고 **코드의 재사용성**이 높아지며 **유지 보수 및 확장성이** 향상된다.

## LangChain에서의 Chain 구성 방식

LangChain은 다음 두 가지 방식을 통해 체인을 구성할 수 있다.

### 1. Off-the-shelf Chains 방식 (클래식 방식)

- LangChain에서 제공하는 **미리 정의된 Chain 클래스**(예: `LLMChain`, `SequentialChain`, `SimpleSequentialChain`)를 활용하는 방식이다.
- 각 클래스는 다양한 chain 알고리즘들을 미리 구현한 것으로 상황에 맞는 것을 선택하여 필요한 구성요소를 전달해 생생한다.
- 이 방식은 LangChain의 **초기 방식**이며, 새로운 기능 확장이나 유연한 구성에 한계가 있기 때문에 현재 **더 이상 사용되지 않음(deprecated)** 상태이다.
  - 현재 LangChain에서는 권장하지 않는 방식이다.

### 2. LCEL (LangChain Expression Language) 방식

- LCEL은 체인을 표현식(Expression) 기반의 선언적 파이프라인 방식으로 구성할 수 있도록 설계된 최신 체인 구성 방법이다. 
- 각 컴포넌트들을 `|` 연산자로 연결하여, 흐름이 자연스럽게 이어지는 형태의 체인을 구성한다.
- LCEL 방식은 간결하고 선언적인 문법을 제공하여 **직관적이고 융통성과 확장성 있는 체인 구성**이 가능하다.
- LCEL은
  - 선형적 흐름 구조를 가진다.
  - 문법이 간결하고 선언적이다.
  - 체인의 구조가 코드만 봐도 쉽게 파악된다.
  - 유연하고 확장성이 매우 뛰어나다.
- `Runnable` 기반 구조
  - LCEL방식을 구성하는 모든 컴포넌트들은 `Runnable` 이라는 공통 인터페이스를 기반으로 동작한다.
  - 체인을 구성하는 각 컴포넌트들은 `Runnable` 을 상속하여 구현하여 이를 통해 일관된 실행 인터페이스를 제공한다.
  - **공통 메소드**:
    - `invoke()`: 단일 입력에 대한 처리
    - `batch()`: 다수 입력을 묶어서 한번에 처리
    - `stream()`: 스트리밍 방식의 요청
    - `ainvoke()`, `abatch()`, `astream()`: 비동기적 처리 메소드

In [1]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
prompt = ChatPromptTemplate.from_template(
    template="{item}에 어울리는 브랜드 이름 {count}개를 만들어 주세요."
)
model = ChatOpenAI(model="gpt-5.4-nano")
parser = StrOutputParser()

In [2]:
query = prompt.invoke({"item":"가방", "count":5})
res = model.invoke(query)
result = parser.invoke(res)
print(result)

1. 루미노백  
2. 모던루프 백  
3. 벨라트레일  
4. 노바킨드 백컴퍼니  
5. 데이브리즈 가방


In [3]:
############################################################
# 기존의 Off the shell 방식 - langchain-classic 설치 필요
############################################################
from langchain_classic import LLMChain
# chain을 구성하는 요소들을 넣어서 생성.
# prompt_template -[prompt]-> model -[응답]-> output parser -> 최종결과
chain = LLMChain(
    prompt=prompt,
    llm=model,
    output_parser=parser
)

res = chain.invoke({"item":"가방", "count":3})
print(res)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_4200\2219368803.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(


{'item': '가방', 'count': 3, 'text': '1. 루나노트(LunaNote)  \n2. 코지마린(CoziMarine)  \n3. 루프앤룩(Loop&Look)'}


In [4]:
##############################
#  LCEL
##############################
chain2 = prompt | model | parser
print(type(chain2))
res2 = chain2.invoke({"item":"TV 브랜드", "count":3}) 

<class 'langchain_core.runnables.base.RunnableSequence'>


In [5]:
print(res2)

아래는 TV 브랜드에 어울리는 느낌의 브랜드 이름 3개입니다(예: 발음이 쉽고, 기술/영상 이미지를 연상하도록 구성).

1. **선명비전(鮮明Vision)**
2. **블루라이트(BluLight)**
3. **프로스코프(ProScope)**


# Runnable 타입 주요 클래스


## [Runnable](https://reference.langchain.com/python/langchain_core/runnables/#langchain_core.runnables.base.Runnable)
- LangChain의 Runnable은 실행 가능한 작업 단위를 캡슐화한 개념으로, 데이터 흐름의 각 단계를 정의하고 **체인(chain) 에 포함 되어**  복잡한 작업의 각 단계를 수행 한다.
- **Chain을 구성하는 class들**은 Runnable의 상속 받아 구현한다.
- **Prompt Template클래스**, **Chat 모델**, **Output Parser 클래스** 등 다양한 컴포넌트가 Runnable을 상속받아 구현된다.

### 주요 특징
- 작업 단위의 캡슐화:
    - Runnable은 특정 작업(예: 프롬프트 생성, LLM 호출, 출력 파싱 등)을 수행하는 독립적인 컴포넌트이다.
    - 각 컴포넌트는 독립적으로 테스트 및 재사용이 가능하며, 조합하여 복잡한 체인을 구성할 수 있다.
- 체인 연결 및 작업 흐름 관리:
    - Runnable은 체인(chain, 일련의 연결된 작업 흐름)을 구성하는 기본 단위로 사용된다.
    - LangChain Expression Language(LCEL)를 사용하면 | 연산자를 통해 여러 Runnable을 쉽게 연결할 수 있다.
    - 입력과 출력의 형식을 일관되게 유지하여 각 단계가 자연스럽게 연결된다.
- 모듈화 및 디버깅 용이성:
    - 각 단계가 명확히 분리되어 문제 발생 시 어느 단계에서 오류가 발생했는지 쉽게 확인할 수 있다.
    - 복잡한 작업을 작은 단위로 나누어 체계적으로 관리할 수 있다.
      
### Runnable의 표준 메소드
- 모든 Runnable이 구현하는 공통 메소드
    - **`invoke(input, config:RunnableConfig)->output`**: 단일 입력을 처리하여 결과를 반환.
    - **`batch(input:list, config:RunnableConfig|list[RunnableConfig]) -> list[Output]`**: 여러 입력 데이터들을 한 번에 처리.
    - **`stream(input, config:RunnableConfig) -> Iterator[Output]`**: 입력에 대해 스트리밍 방식으로 응답을 반환.
    - **`assign(**kwargs)`**:
      -  앞 Runnable의 출력 결과에 새로운 key–value 쌍의 Field 추가(assign) 하여 다음 Runnable로 전달.
      -  값으로는 Runnable 객체(LCEL체인등)나 고정 값(리터럴) 모두 가능하며, 각 항목은 실행 시 평가되어 기존 출력에 병합한다.
      -  주로 앞 단계의 출력에 부가 정보(field)를 추가하고자 할 때 사용한다. 특히 `RunnablePassthrough`와 결합해, 입력을 그대로 넘기면서 특정 field만 추가할 때 자주 사용


In [6]:
from langchain_core.runnables import Runnable
print(isinstance(model, Runnable), isinstance(prompt, Runnable), isinstance(parser, Runnable), isinstance(chain2, Runnable))

True True True True


### Runnable의 주요 구현체(하위 클래스)

- 다음 클래스들은 기능을 제공하는 것이 아니라 **chain 구조를 다양하게 구성** 할 수 있도록 도와주는 **Runnable** 타입의 클래스들이다.

- **`RunnableSequence`**
    - 여러 `Runnable`을 순차적으로 연결하여 실행하는 구성이다.
    - 각 단계의 출력이 다음 단계의 입력으로 전달된다.
    - 보통은 LCEL 문법을 사용해서 정의한다.
      - LCEL을 사용하여 체인을 구성할 경우 자동으로 `RunnableSequence`로 변환된다.


In [7]:
from langchain_core.runnables import RunnableSequence

# chain = RunnableSequence(prompt, model, parser)
chain = prompt | model | parser
#chain
chain.invoke({"item":"물", "count":2})

'1) **물빛다움**  \n2) **청수향기**'


- **`RunnableLambda`**
    - Lambda 표현식의 함수를 `Runnable`로 변환할 때 사용한다.    
    - 일반함수도 `RunnableLambda`로 변환할 수 있다. 단 일반 함수는 변환 없이 chain에 포함 시킬 수 있기 때문에 굳이 변환할 필요가 없다.
    - Runnable로 만들 함수 구문
        - parameter: 입력 값 1개선언.
        - return: 다음 chain에 전달할 값의 형식


In [12]:
from langchain_core.runnables import RunnableLambda

# RunnableLambda(함수)
# 함수 - 파라미터 (1개 -> 앞 체인으로 부터 받은 값에 맞춘다.)
#   - 리턴값 -  > 다음 체인의 입력 타입에 맞게 반환.
c1 = RunnableLambda(lambda input_data:f"{input_data}에 대해서 한 문장으로 설명해줘.")
# type(c1)
c1.invoke("LLM 모델")

'LLM 모델에 대해서 한 문장으로 설명해줘.'

In [14]:
chain = c1 | model | parser
chain.invoke("LLM 모델") # chain 호출 시에는 첫번째 컴포넌트에 전달할(넣을)값을 넣어서 호출

'LLM(대규모 언어 모델)은 방대한 텍스트 데이터를 학습해 문맥을 이해하고 다음에 올 단어를 예측하는 방식으로 글 생성, 요약, 번역 같은 언어 작업을 수행하는 인공지능 모델입니다.'


-  **`RunnablePassthrough`**
    - 입력 데이터를 가공하지 않고 그대로 다음 단계로 전달하는 `Runnable`이다.
      - 앞 Runnable으로 부터 전달 받은 **입력 값을 다음 Runnable로 그대로 전달**한다.
           - `RunnablePassthrough()`
      - 입력받은 값에 **Field를 추가**해서 전달할 경우 `assign()` 메소드를 사용한다.
           - `RunnablePassthrough.assign(new_key1="new_value1", new_key2="new_value2", ..)`


In [17]:
prompt = ChatPromptTemplate(
    messages = [
        ("system", "모든 응답은 100글자 이내로 작성해줘."),
        ("user","{query}")
    ]
)
model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

# Chain구성 chain 응답: LLM 응답내용, 글자수
chain = prompt | model | parser | RunnableLambda(lambda x : (x, len(x)))


In [18]:
res = chain.invoke("AI에 대해서 설명해줘.")
print(res)

('AI는 사람처럼 학습·판단·생성하는 인공지능입니다. 데이터로 패턴을 익혀 예측합니다.', 47)


In [20]:
def get_value_len(value:str):
    return value, len(value)

# 일반함수(callable)를 chain의 구성으로 포함시킬 수 있다. -> 내부적으로 Runnable로 변환되서 들어간다.
chain2 = prompt | model | parser | RunnableLambda(get_value_len)
chain2.invoke("크리스마스에 대해 설명해줘.")

('크리스마스는 예수 탄생을 기념하는 기독교 축제로, 12월 25일에 지켜집니다.', 43)

In [23]:
from langchain_core.runnables import RunnablePassthrough

rp = RunnablePassthrough() # 단순히 받은 값을 다음으로 통과시킨다.

result = rp.invoke("안녕하세요")
result = rp.invoke([1,2,3,4,5])
result = rp.invoke({"a":10, "b":20})


print(result)

{'a': 10, 'b': 20}


In [24]:
# 입력받은 값(Dictinary)에 item을 추가해서 다음으로 전달.
r1 = RunnableLambda(lambda x: "서울시 금천구 독산동")
r2 = RunnableLambda(lambda x: "010-1111-2222")

# assign(Key= Runnable, ...)
## 입력받은 딕셔너리에 address와 tel_no key를 추가. value는 Runnable을 호출해서 반환된 값을 설정.
rp2 = RunnablePassthrough.assign(
    address = r1,
    tel_no= r2
)
result = rp2.invoke({"name":"홍길동"})
result

{'name': '홍길동', 'address': '서울시 금천구 독산동', 'tel_no': '010-1111-2222'}


- **`RunnableParallel`**
    - 여러 `Runnable`을 병렬로 실행한 후, 결과를 결합하여 다음 단계로 전달한다.
    - 
        ```python
        RunnableParallel(
            {
                "key1":Runnable1, 
                "key2":Runnable2,
                "key3":Runnable3, ...
            }
        )
        ```
    - 각 Runnable의 실행결과를 Value로 Dictionary를 생성해서 반환한다.
    - LCEL로 정의할 때는 Chain에 dictionary로 정의한다.



In [28]:
from langchain_core.runnables import RunnableParallel

r1 = RunnableLambda(lambda x: x + 10)
r2 = RunnableLambda(lambda x: x - 10)
r3 = RunnableLambda(lambda x: x * 10)
r4 = RunnableLambda(lambda x: x / 10)

# c = r1 | r2 | r3 | r4

parallel = RunnableParallel(
    {
        "value1": r1,
        "value2": r2,
        "value3": r3,
        "value4": r4,
        "org_value": RunnablePassthrough() # 입력받은 값을 그대로 다음으로 넘겨줘야 할 경우에 사용.
    }
)

result = parallel.invoke(200)

In [29]:
result

{'value1': 210,
 'value2': 190,
 'value3': 2000,
 'value4': 20.0,
 'org_value': 200}

In [30]:
c = RunnablePassthrough() | { # RunnableParallel() 생략가능
        "value1": r1,
        "value2": r2,
        "value3": r3,
        "value4": r4,
        "org_value": RunnablePassthrough() # 입력받은 값을 그대로 다음으로 넘겨줘야 할 경우에 사용.
    }
c.invoke(2000)

{'value1': 2010,
 'value2': 1990,
 'value3': 20000,
 'value4': 200.0,
 'org_value': 2000}

### LCEL Chain 예제

In [ ]:
##################################################
# TODO 1
# 음식 이름을 입력하면 그 음식의 레시피를 llm이 출력하는 Chain을 LCEL 을 이용해서 구성한다.
# 입력 : 음식 이름 - recipe_chain.invoke({"food":"김치찌게"})
# 출력 : 음식의 레시피 - 김치찌게 레시피. 

# chain구성: prompt_template -> model(gpt-5-mini) -> StrOutputParser

In [36]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

system_prompt = """
<instruction>
당신은 숙련된 요리전문 AI Assistant입니다.
요청받은 음식의 레시피를 자세하고 쉽게 작성해 주세요.
출력 방법은 아래 output_format을 참고해서 응답해주세요.
</instruction>

<output_format>
- Markdown 형식으로 답변을 작성합니다.
- 응답 내용에는 다음 항목들을 포함합니다.
    - 요리 이름
    - 요리 기본정보
        - 난이도
        - 조리시간
        - 인분
    - 요리에 필요한 재료
    - 요리 방법
    - 팁
</output_format>
"""
prompt = ChatPromptTemplate(
    messages = [
        ("system", system_prompt),
        ("user", "{food}의 레시피를 작성해 주세요.")
    ]
)
model = ChatOpenAI(model="gpt-5.4-nano")
parser = StrOutputParser()

In [31]:
prompt = ChatPromptTemplate.from_template(
    template="{food}의 레시피를 알려주세요."
)
model = ChatOpenAI(model="gpt-5.4-nano")
parser = StrOutputParser()

In [37]:
recipe_chain = prompt | model | parser

res = recipe_chain.invoke({"food":"김치찌개"})

print(res)

## 요리 이름
**김치찌개**

---

## 요리 기본정보
- **난이도**: 중급(기본기만 있으면 쉬워요)
- **조리시간**: 약 **30~40분**
- **인분**: **3~4인분**

---

## 요리에 필요한 재료
### 기본 재료
- **신김치** 1~1.5컵 (약 250~300g), 먹기 좋게 썰기  
- **돼지고기(앞다리/삼겹/목살)** 200~250g (한입 크기)
- **두부** 1/2모 (150~200g), 깍둑썰기
- **양파** 1/2개 (채 썰기)
- **대파** 1/2대 (송송)
- **마늘** 1~2쪽 (다지기)
- **고추가루** 1~2큰술 (선택이지만 맛의 핵심)
- **물** 500~700ml (취향에 따라 조절)
- **된장** 1/2작은술 (선택, 감칠맛)
- **소금** 약간 (간은 마지막에 조절)
- **후추** 약간
- **참기름** 1작은술 (선택, 마무리 향)

### 육수/양념(선택)
- **멸치 다시마 육수** 또는 **시판 국물용 육수** 있으면 더 좋아요(물 대신 사용 가능)
- **고춧가루+김치국물**로도 충분히 맛있게 낼 수 있어요

---

## 요리 방법
1. **돼지고기 손질**
   - 돼지고기를 한입 크기로 썰어 준비합니다.
   - (선택) 핏물을 5분 정도 가볍게 빼두면 더 깔끔해요.

2. **김치 볶기**
   - 냄비에 **기름을 아주 약간** 두르고(또는 돼지고기 기름으로 진행해도 OK),
   - **신김치**를 넣어 **2~3분** 볶습니다.
   - 이때 **고추가루(1큰술 정도)**와 **마늘**을 함께 넣으면 향이 더 좋아요.
   - 김치가 타지 않게 중불로 가볍게 볶아주세요.

3. **고기 넣고 끓이기**
   - 볶아둔 김치에 **돼지고기**를 넣고 2분 정도 더 볶습니다.
   - 고기가 겉면에 색이 올라오면 **물(500~700ml)**을 붓습니다.
   - **센불**로 끓이다가, 끓기 시작하면 **중불~약불**로 줄입니다.

4. **양파 넣기**
   

In [38]:
from IPython.display import Markdown

Markdown(res)

## 요리 이름
**김치찌개**

---

## 요리 기본정보
- **난이도**: 중급(기본기만 있으면 쉬워요)
- **조리시간**: 약 **30~40분**
- **인분**: **3~4인분**

---

## 요리에 필요한 재료
### 기본 재료
- **신김치** 1~1.5컵 (약 250~300g), 먹기 좋게 썰기  
- **돼지고기(앞다리/삼겹/목살)** 200~250g (한입 크기)
- **두부** 1/2모 (150~200g), 깍둑썰기
- **양파** 1/2개 (채 썰기)
- **대파** 1/2대 (송송)
- **마늘** 1~2쪽 (다지기)
- **고추가루** 1~2큰술 (선택이지만 맛의 핵심)
- **물** 500~700ml (취향에 따라 조절)
- **된장** 1/2작은술 (선택, 감칠맛)
- **소금** 약간 (간은 마지막에 조절)
- **후추** 약간
- **참기름** 1작은술 (선택, 마무리 향)

### 육수/양념(선택)
- **멸치 다시마 육수** 또는 **시판 국물용 육수** 있으면 더 좋아요(물 대신 사용 가능)
- **고춧가루+김치국물**로도 충분히 맛있게 낼 수 있어요

---

## 요리 방법
1. **돼지고기 손질**
   - 돼지고기를 한입 크기로 썰어 준비합니다.
   - (선택) 핏물을 5분 정도 가볍게 빼두면 더 깔끔해요.

2. **김치 볶기**
   - 냄비에 **기름을 아주 약간** 두르고(또는 돼지고기 기름으로 진행해도 OK),
   - **신김치**를 넣어 **2~3분** 볶습니다.
   - 이때 **고추가루(1큰술 정도)**와 **마늘**을 함께 넣으면 향이 더 좋아요.
   - 김치가 타지 않게 중불로 가볍게 볶아주세요.

3. **고기 넣고 끓이기**
   - 볶아둔 김치에 **돼지고기**를 넣고 2분 정도 더 볶습니다.
   - 고기가 겉면에 색이 올라오면 **물(500~700ml)**을 붓습니다.
   - **센불**로 끓이다가, 끓기 시작하면 **중불~약불**로 줄입니다.

4. **양파 넣기**
   - 양파를 넣고 **10~15분** 정도 더 끓여주세요.
   - 김치가 충분히 우러나도록 시간을 주는 게 포인트예요.

5. **간 맞추기**
   - 국물이 우러나면 **두부**를 넣기 전에 간을 먼저 봅니다.
   - **된장(선택)**을 약간 넣어 더 깊은 맛을 낼 수 있어요.
   - 짠맛이 강하면 물을 약간 추가하거나 김치를 더 우려내는 방식으로 조절하고,
   - 싱거우면 **소금**을 아주 조금씩만 추가하세요.

6. **두부 넣고 마무리**
   - **두부**를 넣고 **3~5분** 끓입니다(두부가 부서지지 않게).
   - 마지막으로 **대파**, **후추**를 넣고 30초~1분 더 끓입니다.
   - 원하면 **참기름 1작은술**로 마무리하면 풍미가 살아납니다.

---

## 팁
- **신김치가 핵심**: 김치찌개는 숙성된 **신김치**가 맛있어요.  
- **고기 먼저 볶으면 더 진해요**: 김치+고기에서 풍미가 충분히 올라오게 2~3분 볶아주세요.
- **두부는 마지막에**: 오래 끓이면 부서져서 식감이 떨어질 수 있어요.
- **간은 마지막에**: 김치마다 짠맛이 달라서, 소금은 조금씩 조절하세요.
- **더 깊은 맛 내기(선택)**: 멸치 다시마 육수를 쓰거나, 김치 국물을 국물에 한 국자 추가하면 감칠맛이 확 올라갑니다.

원하시면 **돼지고기 대신 햄/참치/소고기**로 바꾼 버전이나, **맵기 조절(순한/진한)** 레시피도 같이 맞춰드릴게요.

In [39]:
res2 = recipe_chain.invoke({"food":"봉골레 파스타"})

In [40]:
Markdown(res2)

## 요리 이름
**봉골레 파스타(Clam Spaghetti)**

---

## 요리 기본정보
- **난이도**: 중급(조개 손질/물기 관리가 포인트)
- **조리시간**: 약 25~35분
- **인분**: 2인분

---

## 요리에 필요한 재료 (2인분)
- 스파게티 면 160~180g
- 조개(바지락 또는 클램) 400g
- 올리브오일 3~4큰술
- 마늘 3~5쪽 (편썰기)
- 페페론치노(또는 청양고추) 1~2개 (취향)
- 화이트와인 100ml *(또는 조개 삶은 물 100ml + 약간의 레몬즙)*
- 파슬리 약간 (다진 것, 있으면 더 좋음)
- 소금 약간 (면 삶기용)
- 후추 약간
- 레몬즙 1~2작은술 *(선택, 비린내 제거/풍미용)*

> **조개 해감이 안 된 제품이라면**: 소금물(굵은소금 기준 약 3%)에 1~2시간 해감 후 사용하세요.

---

## 요리 방법
### 1) 조개 손질 & 해감
1. 조개를 흐르는 물에 가볍게 씻고, 모래가 있을 수 있으면 **해감**을 먼저 합니다.  
2. 해감 후 다시 깨끗이 씻어 물기를 빼주세요.

### 2) 조개 입 열기(소스 베이스 만들기)
1. 팬(또는 냄비)에 **올리브오일 3큰술**을 두르고 중약불로 달굽니다.
2. **편썬 마늘**과 **페페론치노**를 넣고 30초~1분 정도 향을 냅니다. (타지 않게!)
3. **화이트와인**(또는 대체액)을 붓고 살짝 끓입니다.
4. 조개를 넣고 뚜껑을 덮은 뒤 **중불~강불에서 5~8분** 조리합니다.  
   - 조개가 전부 열리면 불을 끕니다.
5. **중요**: 조개에서 나온 국물은 모래가 섞였을 수 있으니,  
   - 체에 한 번 걸러 **모래를 걸러낸 뒤** 사용하세요.

### 3) 면 삶기
1. 큰 냄비에 물을 팔팔 끓이고 **소금 한 꼬집~한 큰술 정도** 넣습니다(간 맞추는 용도).
2. 스파게티를 포장지 지시에 따라 삶되, **면을 1~2분 덜 익혀**(알덴테보다 살짝 덜) 건져둡니다.
3. 면 삶은 물을 **약국자 1~2국자 정도 남겨두세요**.

### 4) 파스타 소스 합치기
1. 조개 소스가 있는 팬에 불을 중간으로 올립니다.
2. 덜 익힌 스파게티를 넣고, **조개 국물 + 면 삶은 물**을 조금씩 추가하며 농도를 맞춥니다.  
3. 소스가 면에 잘 코팅되도록 **30~90초** 빠르게 볶아 마무리합니다.
4. **후추**와 **레몬즙(선택)**, **다진 파슬리**를 넣습니다.
5. 간은 조개 국물의 짠맛이 있으니 **필요할 때만** 소금으로 아주 소폭 조절하세요.

### 5) 서빙
- 접시에 담고 위에 파슬리 또는 후추를 약간 더 뿌려 마무리합니다.

---

## 팁
- **조개 모래 제거**: 해감 + 조개 국물은 꼭 **한 번 체에 거르기**가 핵심입니다.
- **마늘은 타지 않게**: 향만 내고 바로 와인/조개를 넣으세요.
- **농도 조절**: 봉골레는 소스가 너무 걸쭉하면 뻑뻑하고, 너무 묽으면 맛이 약해집니다.  
  → **면 삶은 물을 활용**해 크리미하게 맞추면 좋아요.
- **레몬은 선택**이지만, 조개 특유의 비린 향이 걱정되면 **마지막에 조금** 넣으면 깔끔해집니다.
- **조개가 안 열리면**: 안전을 위해 **사용하지 않는 것이 좋습니다.**

원하시면 **바지락 vs 해감된 클램**, **화이트와인 없이 만드는 버전**, 또는 **좀 더 크리미하게 만드는(버터/생크림 옵션)** 형태로도 조정해 드릴게요.

In [ ]:
##############################################################
#  TODO 2
# 번역할 내용, 번역할 언어 를 입력하면 내용을 그 언어로 번역하는 Chain을 LCEL 을 이용해서 구성한다.
#
## 입력: 번역할 내용, 언어.  translate_chain.invoke({"content":"안녕하세요.", "language":"영어"})
## 출력: "번역할 내용"을 "언어" 로 번역한 결과 - "How are you?".

# chain구성: prompt_template -> model(gpt-5-mini) -> StrOutputParser

In [33]:
prompt = ChatPromptTemplate.from_template(
    template="{content}를 {language}로 번역해줘."
)
model = ChatOpenAI(model="gpt-5.4-nano")
parser = StrOutputParser()

In [34]:
translate_chain = prompt | model | parser

res = translate_chain.invoke({"content":"안녕하세요.", "language":"영어"})

print(res)

“안녕하세요.”는 영어로 **“Hello.”** 또는 상황에 따라 **“Hi.”**라고 번역할 수 있어요.


In [41]:
system_prompt2= """
<instruction>
당신은 다국어가 가능한 숙련된 번역 AI Assistant입니다.
<input_data> 항목에 작성된 내용을 참고 해서 **요청된 문서**의 내용을 **요청된 언어**로 번역해 주세요.
내용의 의미를 해치지 않는 범위에서 최대한 읽기 쉽게 작성해 주세요.
</instruction>

<input_data>
- 번역할 내용: {content}
- 번역할 언어: {language}
</input_data>
"""

prompt_trans = ChatPromptTemplate.from_template(
    template= system_prompt2
)
# prompt.invoke({"content":"안녕하세요" , "language":"영어"})

model_trans = ChatOpenAI(model= "gpt-5.4-mini")

translate_chain = prompt_trans | model_trans | StrOutputParser()

In [44]:
content= """
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data.
"""

In [45]:
res = translate_chain.invoke({"content":content, "language":"한국어"})
print(res)

우세한 시퀀스 변환 모델들은 인코더와 디코더를 포함하는 복잡한 순환 신경망 또는 합성곱 신경망을 기반으로 한다. 가장 뛰어난 성능을 보이는 모델들은 인코더와 디코더를 어텐션 메커니즘으로 연결한다. 우리는 순환과 합성곱을 완전히 배제하고, 오직 어텐션 메커니즘만을 기반으로 하는 새로운 단순한 네트워크 구조인 Transformer를 제안한다. 두 가지 기계번역 과제에 대한 실험 결과, 이 모델들은 더 높은 품질을 보이면서도 병렬화가 더 용이하고 학습 시간도 크게 짧았다. 우리의 모델은 WMT 2014 영어-독일어 번역 과제에서 28.4 BLEU를 달성하여, 앙상블을 포함한 기존 최고 성능을 2 BLEU 이상 상회했다. 또한 WMT 2014 영어-프랑스어 번역 과제에서는 8개의 GPU로 3.5일 동안 학습한 뒤 41.8의 새로운 단일 모델 최고 BLEU 점수를 기록했으며, 이는 문헌에 보고된 최고 성능 모델들의 학습 비용에 비하면 매우 작은 수준이다. 우리는 Transformer가 다른 과제에도 잘 일반화됨을 보이기 위해, 대규모 및 제한된 학습 데이터를 모두 사용하여 영어 구문 분석에 성공적으로 적용하였다.


In [46]:
from langchain_core.runnables import Runnable

isinstance(translate_chain, Runnable)

True

### Chain과 Chain간의 연결

In [ ]:
translate_chain.invoke({"content":"요리레시피", "language":"번역할 언어이름"})

In [ ]:
# 음식 레시피를 원하는 언어로 출력하는 AI Agent가 필요.
## recipe_chain과 translate_chain을 연결
chain = recipe_chain | translate_chain

In [51]:
from operator import itemgetter

ig = itemgetter("language")
ig

a = {"language":"영어", "name":"홍길동"}
ig(a) # 자료구조를 넣어주면 생성할 때 지정한 index/key의 값을 조회해서 반환.

ig2 = itemgetter(2)
b = [10, 20, 30, 40, 50, 60]
ig2(b)


30

In [ ]:
chain = {
    "content":recipe_chain, 
    "language":RunnableLambda(lambda x: x['language'])
}

In [52]:
# 음식 레시피를 원하는 언어로 출력하는 AI Agent가 필요.
## recipe_chain과 translate_chain을 연결
# {}: RunnableParallel
chain = {
    "content":recipe_chain, 
    "language":itemgetter("language")
} | translate_chain

In [53]:
res = chain.invoke({"food":"김치찌개", "language":"독일어"})

In [54]:
print(res)

## Gerichtname
**Kimchi-Jjigae**

---

## Grundinformationen zum Gericht
- **Schwierigkeitsgrad:** Mittel (die richtige Balance der Grundwürzung ist entscheidend)
- **Zubereitungszeit:** ca. 30–40 Minuten
- **Portionen:** 3–4 Portionen

---

## Zutaten

### Hauptzutaten
- **Sauerkimchi (gut gereiftes Kimchi)** 1–1,5 Tassen (ca. 200–250 g)
- **Schweinefleisch (Nacken/Schulter/Bauch usw.)** 200–300 g  
  - (Falls nicht vorhanden: mit zusätzlichem Tofu oder Brühe aus Sardellen/Kelp ersetzbar)
- **Tofu** 1/2 Block (ca. 300 g)
- **Frühlingszwiebel** 1/2 Stange
- **Zwiebel** 1/2 Stück (optional, für mehr Tiefe)
- **Cheongyang-Chili** 1–2 Stück (optional)
- **Wasser** 700–900 ml (kann durch Brühe ersetzt werden)

### Würzung / Geschmack der Suppe
- **Koreanisches Chilipulver (Gochugaru)** 1–2 EL (je nach Schärfe des Kimchis anpassen)
- **Gehackter Knoblauch** 1 EL
- **Koreanische Suppensoyasauce oder normale dunkle Sojasauce** 1–2 EL  
  - (eventuell ist zusätzlich etwas Salz nötig)
- **Zucke

## 함수를 Runnable로 정의하기

### 함수 구현
- **파라미터**
   - 이전 Chain에서 출력한 값을 입력으로 받을 수 있도록 정의한다.
- **리턴값**
   - 다음 Chain으로 입력할 값을 반환하도록 구현한다.

### Runnable 타입으로 만들기
1. LCEL Chain안에 함수를 구성요소로 포함시키면, 그 함수는 자동으로 `Runnable` 로 취급된다.
   - 별도의 래핑이나 추가 처리는 필요하지 않다.
2. `RunnableLambda()` 에 넣어 명시적으로 `Runnable` 타입으로 만든다.
   - Lambda 표현식으로 정의할 경우 `RunnableLambda(lambda 표현식)` 으로 정의해야 한다.
   - 보통 일반함수는 `RunnableLambda`를 사용할 필요 없다.
3. `@chain` decorator를 사용
   - 함수에 `@chain` decorator가 선언되면 그 함수는 `RunnableLambda` 타입이 된다.
   - 이 방식은 LCEL만으로 표현하기 어려운 실행 흐름을 직접 정의해야 할 때 주로 사용된다.
     - LCEL은 순차 실행구조를 따른다. 그래서  제어문을 이용해 그 흐름을 제어할 수가 없다. 
     - 단순한 파이프라인에서는 LCEL만으로도 충분하지만, 다음과 같은 경우에는 한계가 있다.
       - 특정 단계를 조건에 따라 실행하거나 생략해야 하는 경우
       - 동일한 단계를 반복적으로 실행해야 하는 경우
       - 여러 판단 로직에 따라 실행 경로가 달라지는 agent 구조
     - 이처럼 복잡한 업무 흐름을 가지는 agent는 단순한 순차 구조만으로는 원하는 응답 품질을 얻기 어렵다. 결국 실행 흐름 자체를 개발자가 직접 코드로 정의해야 하며, 이러한 경우에 `@chain`을 사용해 chain/agent 함수를 구현한다.
   - 이러한 복잡한 실행 흐름을 보다 구조적으로 정의하기 위해 LangChain에서 추가로 제공하는 것이 **LangGraph**이다.

In [55]:
from langchain_core.runnables import RunnableLambda

def plus(n1, n2):
    return n1 + n2

def wrap_plus(x):
    return plus(x[0], x[1])

# chain = RunnablePassthrough() | RunnableLambda(lambda x: plus(x[0], x[1]))
chain = RunnablePassthrough() | wrap_plus
chain.invoke([1, 2])

3

In [56]:
# recipe_chain과 translate_chain을 이용해서 음식레시피를 특정 언어로 반환.
### recipe_chain의 결과와 translate_chain의 결과 둘을 모두 반환.
from langchain_core.runnables import chain

# RunnableLambda(multi_language_recipe_chain)

@chain
def multi_language_recipe_chain(input_data: dict) -> dict[str, str]:
    food:str = input_data['food'] # 음식 이름
    language:str = input_data['language'] # 레시피 언어
    is_korean:bool = input_data['is_korean'] # 한국어 레시피 필요 여부.

    korean_recipe = recipe_chain.invoke({"food":food})

    result = translate_chain.invoke({"content": korean_recipe, "language": language})

    final_result = {"recipe":result}

    if is_korean: # 한국어 레시피도 요청
        final_result["korean_recipe"] = korean_recipe

    return final_result

type(multi_language_recipe_chain)

langchain_core.runnables.base.RunnableLambda

In [57]:
res = multi_language_recipe_chain.invoke(
    {"food":"돈까스", "language":"중국어", "is_korean":True}
)

In [58]:
res.keys()

dict_keys(['recipe', 'korean_recipe'])

In [59]:
print(res["korean_recipe"])
print(res["recipe"])

## 요리 이름
돈까스 (돼지고기 커틀릿)

---

## 요리 기본정보
- **난이도**: 중급(처리 과정은 간단하지만 튀김 온도/옷 입히기 숙련이 필요)
- **조리시간**: 약 35~50분
- **인분**: 2~3인분

---

## 요리에 필요한 재료
### 돼지고기
- 돼지고기 등심/안심/목살 중 택1: **300~450g** (두께 1~1.5cm)
- 소금: **약간**
- 후추: **약간**
- 밀가루: **적당량**
- 달걀: **1~2개** (풀어서 사용)
- 빵가루(판코 권장): **적당량**

### 튀김용
- 식용유: **튀김에 충분한 양**(깊이는 대개 3~5cm 정도)

### 돈까스 소스(시판 or 홈메이드)
- 시판 소스 사용 시: **원하는 양**
- 홈메이드(선택):  
  - 케첩 4큰술  
  - 우스터소스 2큰술(없으면 간장/굴소스 소량 대체 가능)  
  - 설탕 1큰술  
  - 식초 1큰술  
  - 물 2큰술  
  - 다진 마늘 1작은술(선택)  
  - (선택) 전분물 약간(농도 조절)

### (선택) 곁들임
- 양배추 채: **한 줌~2줌**  
- 소금 약간 + 돈까스 소스(또는 케첩+식초+설탕 약간)

---

## 요리 방법
### 1) 돼지고기 준비하기
1. 돼지고기를 **고르게 두께가 되도록** (약 1~1.5cm) 정리합니다.
2. 양면에 **소금, 후추**를 가볍게 뿌려 밑간합니다.
3. 고기 표면에 밴딩(결 방향을 따라) 칼집을 **약간만** 내주면 뒤틀림이 줄어요.

### 2) 돈까스 옷 입히기(중요!)
1. **밀가루 → 달걀 → 빵가루** 순서로 입힙니다.
2. 바삭함을 위해 빵가루는 **한 번에 끝내지 말고**  
   - 빵가루를 묻힌 뒤 **살짝 눌러 고기가 보이지 않게** 고르게 펴주고  
   - 원하면 **다시 달걀 1번 + 빵가루 1번(더블 코팅)**도 가능합니다.
3. 마무리 후 **5분 정도** 두면(실온) 튀김 시 코팅이 더 안정적으로 붙습니다.

### 3) 소스 만들

# Cache

- 응답 결과를 저장해서 같은 질문이 들어오면 LLM에 요청하지 않고 저장된 결과를 보여주도록 한다.
    - 처리속도와 비용을 절감할 수 있다.
    - 특히 chatbot같이 비슷한 질문을 하는 경우 유용하다.
- 저장 방식은 `메모리`, `sqlite` 등 다양한 방식을 지원한다.
  
    ```python
    set_llm_cache(Cache객체)
    ```

In [65]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache, SQLiteCache

# Cache 설정은 한번만 하면된다.
# set_llm_cache(InMemoryCache())
set_llm_cache(SQLiteCache("cache.sqlite")) # cache를 저장할 파일 경로

In [68]:
res = multi_language_recipe_chain.invoke(
    {"food":"돈까스", "language":"중국어", "is_korean":True}
)

c:\SKN31_\SKN31\.venv\Lib\site-packages\langchain_community\cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]
c:\SKN31_\SKN31\.venv\Lib\site-packages\langchain_community\cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]


In [69]:
print(res["korean_recipe"])
print(res["recipe"])

## 요리 이름
바삭한 돈까스(돼지고기 커틀릿) 레시피

## 요리 기본정보
- **난이도**: 중급(기본 튀김 기술 필요)
- **조리시간**: 약 40~60분
- **인분**: 2~3인분

---

## 요리에 필요한 재료
### 돼지고기 커틀릿
- 돼지고기 등심(또는 안심) 300~450g (돈까스용 두께, 1.5~2cm 내외)
- 소금 1/2작은술
- 후추 약간
- 밀가루(튀김 코팅용) 1/2~1컵
- 달걀 1~2개
- 빵가루(팬코 또는 일반 빵가루) 1~2컵

### 튀김용
- 식용유(튀김에 충분히) 약 500ml 내외

### (선택) 밑간/풍미
- 맛술 1큰술 (선택)
- 다진 마늘 1작은술 (선택)

### 곁들임
- 돈까스 소스
- 양배추 채썰기(또는 샐러드용 야채)
- (선택) 마요네즈, 케첩, 레몬

---

## 요리 방법
### 1) 고기 준비 & 밑간
1. 돼지고기를 돈까스 크기로 준비합니다. (두께가 두꺼우면 반으로 살짝 펴거나 두께를 맞춰주세요.)
2. 양면에 **소금, 후추**를 골고루 뿌립니다.  
   - 원하면 맛술(1큰술)이나 다진 마늘(1작은술)을 함께 섞어 10분 정도 재워도 좋아요.
3. 고기를 **고기 두드리기(선택)**: 너무 두꺼우면 고기망치로 살짝 두드려 두께를 고르게 하면 익는 속도가 좋아집니다.

### 2) 코팅(튀김 옷 입히기)
1. 준비물 3가지를 **순서대로** 준비합니다:  
   - 밀가루 → 달걀 → 빵가루
2. 고기에 **밀가루를 얇게 먼저** 묻힙니다. (표면이 마르면 달걀이 잘 안 붙어요.)
3. 달걀에 **충분히 적셔** 코팅합니다.
4. 마지막으로 빵가루를 **눌러가며** 묻힙니다.  
   - 바삭함을 원하면 빵가루를 손으로 살짝 “단단히” 붙여주세요.
5. 코팅을 끝낸 돈까스는 가능하면 **5~10분 정도** 잠깐 휴지시켜요(코팅이 더 잘 붙습니다).

### 3) 튀김
1. 팬 또는 냄비에 식용유를 **돈까스가 2/3 이상 잠길 정도**로 두릅니다.  
2. 기름 온도는 대략 **